In [ ]:
import typing as t
from dataclasses import dataclass
from pydantic import BaseModel, RootModel, create_model

T = t.TypeVar("T")

PARAMETERS = {
    "value": 5
}

class Parameter(BaseModel, t.Generic["T"]):
    default: "T"
    key: str

    def model_serialiser(self):
        ...
        # check the extra parameters and if there is a parameter called parameterise=True
        # then try see if key in PARAMETERS else return self.default for now (ill make it more complex later just as a poc)

class ConfigModelValue(RootModel, t.Generic["T"]):
    root: T
    _RootClass: t.Type[BaseModel]

    def build(self):
        return self._RootClass.model_validate(self.model_dump(context={"serialise": True}))

@dataclass
class ConfigModel(t.Generic["T"]):
    value: ConfigModelValue[T]

    def __getattribute__(self, name):
        assert name in root...
        return ConfigSelector(self, (name,))


@dataclass(slots=True)
class ConfigSelector:
    config_model: ConfigModel
    path: t.Tuple[str,...]

    def set_as_parameter(self, key):
        # 1 replace the schema of the ConfigModelValue with ParameterTemplate at the appropriate place
        # 2 rebuild the model
        # 3 set ConfigModel.value = RebuiltConfigModelValue(...)


In [ ]:
class SubModel(BaseModel):
    a: str
    b: float
    c: int

class TestModel(BaseModel):
    a: int
    b: SubModel


t = TestModel(a=1,b=SubModel(a='a', b=1.1, c=2))

c = ConfigModel.from_class(t)

c.b.c.set_as_parameter("value")
c.b.a.set_as_parameter("str_value")

c.model_dump -> {
    "a": 1, 
    "b": {"a": {"key": "str_value", "default": 'a'}, "b": 1.1, "c": {"key": "value", "default": 1}}
}

c.build().model_dump -> {
    "a": 1, 
    "b": {"a": 'a', "b": 1.1, "c": 5}
}